In [0]:
dbutils.widgets.text("catalog_name",    "food_inspection")
dbutils.widgets.text("bronze_schema",   "bronze")
dbutils.widgets.text("silver_schema",   "silver")
dbutils.widgets.text("metadata_schema", "metadata")
 
catalog_name    = dbutils.widgets.get("catalog_name")
bronze_schema   = dbutils.widgets.get("bronze_schema")
silver_schema   = dbutils.widgets.get("silver_schema")
metadata_schema = dbutils.widgets.get("metadata_schema")
 
spark.sql(f"USE CATALOG {catalog_name}")
print(f"Silver pipeline — {catalog_name}")

In [0]:
from pyspark.sql.functions import (
    col, lit, trim, upper, lower, when, split, explode, posexplode,
    regexp_extract, regexp_replace, to_date, concat_ws, md5, length,
    current_timestamp, current_date, coalesce, size, count as spark_count,
    row_number, monotonically_increasing_id, max as spark_max
)
from pyspark.sql.window import Window
from pyspark.sql import DataFrame
from functools import reduce
from datetime import datetime
 
def log_dqx(table_name, city, total, passed, failed):
    spark.sql(f"""
        INSERT INTO {catalog_name}.{metadata_schema}.dqx_execution_log VALUES (
            '{table_name}', '{city}', current_timestamp(),
            {total}, {passed}, {failed}, current_date()
        )
    """)

In [0]:
#loading dallas bronze 
dallas_bronze = spark.table(f"{catalog_name}.{bronze_schema}.bronze_dallas")
dal_source_count = dallas_bronze.count()
print(f"Bronze Dallas rows: {dal_source_count}")

In [0]:
# drop the full duplicate rows
dal_core_cols = [c for c in dallas_bronze.columns if not c.startswith("_")]
dallas = dallas_bronze.dropDuplicates(dal_core_cols)
dal_after_dedup = dallas.count()
dal_dupes_dropped = dal_source_count - dal_after_dedup
 
print(f"Full duplicate rows dropped: {dal_dupes_dropped} ({dal_source_count} → {dal_after_dedup})")

In [0]:
#rename to snake_case
rename_map = {
    "Restaurant_Name":   "restaurant_name",
    "Inspection_Type":   "inspection_type",
    "Inspection_Date":   "inspection_date_raw",
    "Inspection_Score":  "inspection_score",
    "Street_Number":     "street_number",
    "Street_Direction":  "street_direction",
    "Street_Name":       "street_name",
    "Street_Type":       "street_type",
    "Street_Unit":       "street_unit",
    "Street_Address":    "street_address",
    "Zip_Code":          "zip_code",
    "Inspection_Month":  "inspection_month",
    "Inspection_Year":   "inspection_year",
    "Lat_Long_Location": "lat_long_location",
}
 
for old, new in rename_map.items():
    if old in dallas.columns:
        dallas = dallas.withColumnRenamed(old, new)
 
print(f"Core columns renamed")

In [0]:
#type casting
dallas = (dallas
    .withColumn("inspection_date", to_date(col("inspection_date_raw"), "MM/dd/yyyy"))
    .withColumn("inspection_score", col("inspection_score").cast("int"))
    .withColumn("zip_code", trim(col("zip_code")).cast("string"))
    .drop("inspection_date_raw")
)
 
print("Types cast: inspection_date→DATE, inspection_score→INT, zip→STRING")

In [0]:
#trim whitespaces in string column names 
dal_string_cols = ["restaurant_name", "inspection_type", "street_number",
                   "street_direction", "street_name", "street_type", "street_unit",
                   "street_address"]
 
for c in dal_string_cols:
    if c in dallas.columns:
        dallas = dallas.withColumn(c, trim(col(c)))
 
print(f"Trimmed {len(dal_string_cols)} string columns")

In [0]:
#clamp negativge inspection scores to 0
dal_negative = dallas.filter(col("inspection_score") < 0).count()
print(f"Negative scores clamped to 0: {dal_negative}")
 
dallas = dallas.withColumn("inspection_score",
    when(col("inspection_score") < 0, 0).otherwise(col("inspection_score"))
)

In [0]:
# B8: Parse lat/long — keep ALL coordinates, flag out-of-range
dallas = (dallas
    .withColumn("latitude",
        regexp_extract(col("lat_long_location"), r"\(([-0-9.]+),", 1).cast("double"))
    .withColumn("longitude",
        regexp_extract(col("lat_long_location"), r",\s*([-0-9.]+)\)", 1).cast("double"))
    .withColumn("is_coords_out_of_range",
        when(col("latitude").isNotNull() & ~col("latitude").between(32.0, 34.0), True)
        .otherwise(False))
)

dal_out_of_range_coords = dallas.filter(col("is_coords_out_of_range") == True).count()
dal_null_coords = dallas.filter(col("latitude").isNull()).count()
print(f"Coordinates: {dal_out_of_range_coords} out of range (flagged), {dal_null_coords} null (no location data)")

In [0]:
#flagging out of area zipcodes
dallas = dallas.withColumn("is_out_of_area",
    when(~col("zip_code").rlike("^(750|751|752|753|754|755|760|761|762)"), lit(True))
    .otherwise(lit(False))
)
 
dal_out_of_area = dallas.filter(col("is_out_of_area") == True).count()
print(f"Out-of-DFW-area records flagged: {dal_out_of_area}")

In [0]:
dallas = dallas.withColumn("inspection_id",
    md5(concat_ws("||",
        coalesce(trim(col("restaurant_name")), lit("")),
        coalesce(col("inspection_date").cast("string"), lit("")),
        coalesce(trim(col("street_address")), lit("")),
        coalesce(trim(col("inspection_type")), lit("")),
        coalesce(col("inspection_score").cast("string"), lit(""))
    ))
)

In [0]:
#row number tie breaker for residual collisions
w = Window.partitionBy("inspection_id").orderBy(col("_ingestion_timestamp"))
dallas = (dallas
    .withColumn("_row_num", row_number().over(w))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)
 
dal_after_tiebreaker = dallas.count()
dal_tiebreaker_dropped = dal_after_dedup - dal_after_tiebreaker
print(f"Tiebreaker resolved {dal_tiebreaker_dropped} residual collisions ({dal_after_dedup} → {dal_after_tiebreaker})")

In [0]:
#Count violations per inspection (for VR-008 and fact table)
violation_desc_cols = [c for c in dallas.columns if "Violation_Description" in c]
 
density_expr = sum(
    when(col(f"`{c}`").isNotNull() & (trim(col(f"`{c}`").cast("string")) != ""), 1).otherwise(0)
    for c in violation_desc_cols
)
 
dallas = dallas.withColumn("n_violations", density_expr)
 
print(f"Violation count computed from {len(violation_desc_cols)} description columns")

In [0]:
# Build quarantine reason
dallas = dallas.withColumn("_quarantine_reason", lit(None).cast("string"))
 
# VR-001: restaurant_name cannot be null
dallas = dallas.withColumn("_quarantine_reason",
    when(col("restaurant_name").isNull() | (trim(col("restaurant_name")) == ""),
        lit("restaurant_name_null"))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-002: inspection_date cannot be null
dallas = dallas.withColumn("_quarantine_reason",
    when(col("inspection_date").isNull(),
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("inspection_date_null")), lit("inspection_date_null")))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-003: inspection_type cannot be null
dallas = dallas.withColumn("_quarantine_reason",
    when(col("inspection_type").isNull() | (trim(col("inspection_type")) == ""),
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("inspection_type_null")), lit("inspection_type_null")))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-004: zip_code must be valid 5-digit format
dallas = dallas.withColumn("_quarantine_reason",
    when(col("zip_code").isNull() | ~col("zip_code").rlike("^\\d{5}(-\\d{4})?$"),
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("zip_invalid")), lit("zip_invalid")))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-005: inspection_score cannot exceed 100
dallas = dallas.withColumn("_quarantine_reason",
    when(col("inspection_score") > 100,
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("score_over_100")), lit("score_over_100")))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-008: score >= 90 cannot have > 3 violations — ASSIGNMENT RULE
dallas = dallas.withColumn("_quarantine_reason",
    when((col("inspection_score") >= 90) & (col("n_violations") > 3),
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("vr008_score90_over3_violations")), lit("vr008_score90_over3_violations")))
    .otherwise(col("_quarantine_reason"))
)
 
print("Quarantine reasons tagged")

In [0]:
# Split into clean and quarantine
dallas_quarantine = dallas.filter(col("_quarantine_reason").isNotNull())
dallas_clean = dallas.filter(col("_quarantine_reason").isNull()).drop("_quarantine_reason")
 
dal_passed = dallas_clean.count()
dal_failed = dal_after_tiebreaker - dal_passed
 
print(f"Dallas DQX: {dal_after_tiebreaker} total → {dal_passed} passed, {dal_failed} quarantined")

In [0]:
# Write quarantine table
(dallas_quarantine.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{silver_schema}.quarantine_dallas"))
 
print(f"Quarantine table written: {dallas_quarantine.count()} rows")
display(dallas_quarantine.groupBy("_quarantine_reason").count().orderBy(col("count").desc()))

In [0]:
#Write silver.dallas_inspections
# Select core inspection columns + all violation columns
dal_inspection_cols = [
    "inspection_id", "restaurant_name", "inspection_type", "inspection_date",
    "inspection_score", "street_address", "zip_code",
    "street_number", "street_direction", "street_name", "street_type", "street_unit",
    "latitude", "longitude", "inspection_month", "inspection_year",
    "n_violations", "is_out_of_area"
]

# Also keep all violation columns (they stay wide for now)
all_violation_cols = [c for c in dallas_clean.columns if c.startswith("Violation")]
dal_final_cols = dal_inspection_cols + all_violation_cols
 
dallas_inspections = dallas_clean.select(dal_final_cols)
 
(dallas_inspections.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{silver_schema}.dallas_inspections"))
 
dal_insp_count = spark.table(f"{catalog_name}.{silver_schema}.dallas_inspections").count()
print(f"silver.dallas_inspections: {dal_insp_count} rows")

In [0]:
# B15: Unpivot all 25 violation slots into rows
# Handle column 20 double-space typo defensively

union_dfs = []
for n in range(1, 26):
    desc_col   = f"Violation_Description___{n}"
    points_col = f"Violation_Points___{n}"
    detail_col = f"Violation_Detail___{n}"
    memo_col   = f"Violation_Memo___{n}"
    
    # Column 20 has known double-space typo
    if memo_col not in dallas_clean.columns:
        alt = f"Violation__Memo___{n}"
        if alt in dallas_clean.columns:
            memo_col = alt
    
    if desc_col not in dallas_clean.columns:
        continue
    
    cols_to_select = [col("inspection_id")]
    cols_to_select.append(col(f"`{desc_col}`").cast("string").alias("violation_description"))
    
    if points_col in dallas_clean.columns:
        cols_to_select.append(col(f"`{points_col}`").cast("double").alias("violation_points"))
    else:
        cols_to_select.append(lit(None).cast("double").alias("violation_points"))
    
    if detail_col in dallas_clean.columns:
        cols_to_select.append(col(f"`{detail_col}`").cast("string").alias("violation_detail"))
    else:
        cols_to_select.append(lit(None).cast("string").alias("violation_detail"))
    
    if memo_col in dallas_clean.columns:
        cols_to_select.append(col(f"`{memo_col}`").cast("string").alias("violation_memo"))
    else:
        cols_to_select.append(lit(None).cast("string").alias("violation_memo"))
    
    part = (dallas_clean.select(cols_to_select)
        .withColumn("violation_code", lit(str(n)))
        .filter(col("violation_description").isNotNull() & (trim(col("violation_description")) != ""))
    )
    
    union_dfs.append(part)

dallas_violations = reduce(DataFrame.unionByName, union_dfs)
print(f"Unpivoted all 25 violation slots: {dallas_violations.count()} rows")

In [0]:
#handle dallas inspections with zero violations
dal_inspections_with_violations = dallas_violations.select("inspection_id").distinct()
dal_all_inspections = dallas_clean.select("inspection_id")
 
dal_no_violation_ids = dal_all_inspections.join(
    dal_inspections_with_violations, "inspection_id", "left_anti"
)
 
dal_no_viol_count = dal_no_violation_ids.count()
print(f"Dallas inspections with zero violations: {dal_no_viol_count}")
 
dal_default_violations = (dal_no_violation_ids
    .withColumn("violation_code", lit("NO_VIOLATION"))
    .withColumn("violation_description", lit("No violation recorded"))
    .withColumn("violation_detail", lit(None).cast("string"))
    .withColumn("violation_memo", lit(None).cast("string"))
    .withColumn("violation_points", lit(None).cast("double"))
)
 
dallas_violations = dallas_violations.select(
    "inspection_id", "violation_code", "violation_description",
    "violation_detail", "violation_memo", "violation_points"
).unionByName(dal_default_violations)

In [0]:
# Write violations table
(dallas_violations.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{silver_schema}.dallas_violations"))
 
dal_viol_final = spark.table(f"{catalog_name}.{silver_schema}.dallas_violations").count()
print(f"silver.dallas_violations: {dal_viol_final} rows (incl. {dal_no_viol_count} 'No Violation' defaults)")

In [0]:
# Log Dallas DQX
log_dqx("dallas_inspections", "Dallas", dal_source_count, dal_passed, dal_failed)
print(f"Dallas logged — total: {dal_source_count}, passed: {dal_passed}, quarantined: {dal_failed}")